# Case 1: 100 replications for main text Figure 2(b)

Generate and feature-standardize 100 datasets with N=2000, D=20, K*=10, and
component standard deviation 1, using seeds 0--99. For each replicate,
fit K-means across K=2,...,35 once, reuse these partitions for every score,
and select r from the grid 5,10,...,130 using Algorithm 1.
Evaluate Spearman correlations only on K=2,...,25.

Each boxplot value is a replicate's rNN correlation minus its ordinary
Silhouette correlation. Figure 2(b) uses ACC; this notebook also saves
ARI/NMI diagnostics. `combined_boxplot_figure.ipynb` reads the saved
replication results to assemble Figure 2.


In [ ]:
# -- Core numerical stack --
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

# -- Data generation, standardization, clustering, and external validity metrics --
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# -- k-nearest-neighbor queries used to build the rNN affinity distance matrices --
from sklearn.neighbors import NearestNeighbors

# -- Plotting (final boxplot figures) --
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## Helper functions

In [ ]:
def simulate_gaussian_data(n_samples=2000, n_features=20, K_star=10,
                           cluster_std=1, random_state=1):
    """Generate raw Case 1 Gaussian-mixture data (main text Section 4.1).
    
    The study uses N=2000, D=20, K*=10 and component standard deviation 1.
    Centers are drawn uniformly from [-10, 10]^20 by make_blobs. Seed 1
    defines the representative dataset; seeds 0--99 define the replications.
    Return raw features. The representative plots use them directly;
    the 100-replication boxplot caller standardizes each generated dataset."""
    # Main text Section 4.1: draw centers uniformly from [-10,10]^20 and
    # X_i | y_i=c ~ N(mu_c, I_20). With N=2000 and K*=10, each component has 200 points.
    # Changing random_state redraws the centers, observations, and their row order.
    X, y = make_blobs(n_samples=n_samples, n_features=n_features,
                      centers=K_star, cluster_std=cluster_std,
                      random_state=random_state)
    return X, y


def clustering_accuracy(y_true, y_pred):
    """Clustering accuracy via the Hungarian algorithm.

    Finds the best 1-to-1 matching between predicted cluster labels and
    true class labels (since predicted labels are arbitrary integers) by
    maximizing the matched point count with linear_sum_assignment, then
    reports that best count as a fraction of all points.
    """
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    # ACC = max over one-to-one label matchings of (matched observations)/N.
    # The assignment solver minimizes cost, so negation or max-count subtraction
    # converts maximum overlap to its cost convention. Unmatched groups earn no credit.
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return w[row_ind, col_ind].sum() / y_pred.size


def rnn_affinity_distance(X, r):
    """Build the binary-neighbor distance used by rNN-Silhouette.
    
    Query r+1 neighbors and discard the first returned match. For distinct
    observations this leaves r other points. Take the symmetric union of
    directed neighbor links and return 1-A with a zero diagonal. Reuse this
    distance matrix across candidate cluster counts for the same dataset."""
    n = X.shape[0]
    # For distinct observations, dropping the first match removes self.
    nbrs = NearestNeighbors(n_neighbors=r + 1, algorithm='auto').fit(X)
    dists, idx = nbrs.kneighbors(X)
    # With distinct points, column 0 is the query itself, leaving r other neighbors.
    # For coincident points, the neighbor library determines the order of zero-distance ties.
    dists, idx = dists[:, 1:], idx[:, 1:]

    # Section 3.1: first form the directed binary affinity B_ij = 1{j in N_r(i)}.
    # Neighbor search uses features only; neither the reference labels nor K enter B.
    A = np.zeros((n, n))
    rows = np.arange(n)[:, None]
    A[rows, idx] = 1.0

    # Symmetric union: A_ij = max(B_ij, B_ji) = 1{j in N_r(i) OR i in N_r(j)}.
    # An edge in either direction is enough; this is not a mutual-neighbor graph.
    A = np.maximum(A, A.T)   # symmetrise (OR rule)
    np.fill_diagonal(A, 0)
    # The affinity is already binary. This retained min-max step leaves its 0/1 values unchanged.
    a_min, a_max = A.min(), A.max()
    if a_max > a_min:
        A = (A - a_min) / (a_max - a_min)
    # The paper uses dissimilarities: d_r(i,j)=1-A_ij off the diagonal, d_r(i,i)=0.
    # Thus linked points have distance 0 and unlinked points have distance 1.
    # Pass this N-by-N matrix as metric="precomputed", not as N feature vectors.
    D = 1.0 - A
    np.fill_diagonal(D, 0)
    return D


def spearmanr_safe(a, b):
    """Spearman correlation, returns NaN on degenerate input."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    # Compare paired entries only. In the experiments an entry is a candidate
    # partition (one K), not an observation; SciPy averages tied ranks.
    mask = ~np.isnan(a) & ~np.isnan(b)
    if mask.sum() < 2:
        return np.nan
    rho, _ = spearmanr(a[mask], b[mask])
    return float(rho)


def select_r_topK_spearman(sil_dict, r_values, m, n_k, threshold=0.95):
    """Apply Algorithm 1 to score curves ordered by increasing K.
    
    Correlate the last m_eff scores with a strictly decreasing sequence,
    where m_eff = max(2, min(m, n_k - 1)). Choose the smallest candidate
    whose correlation reaches the threshold; if none does, choose a maximizer
    (the smallest candidate breaks ties). Reference labels and external
    correlations are not used here. Return the selected neighborhood size.
    The high-K window can overlap the evaluation range."""
    # Algorithm 1 scans candidates in increasing order. Here m is the paper's
    # requested window size s, and threshold is tau; n_k counts candidate K values.
    avail = sorted(r for r in r_values if r in sil_dict)
    if not avail:
        return r_values[0]
    # Implemented window convention: keep at most n_k-1 entries (at least two).
    # For K=2,...,35 and m=35 this gives 33 entries, K=3,...,35.
    # This clipping differs by one from the paper's K_lo formula when s spans
    # the entire grid; retain it here to preserve the existing numerical results.
    m_eff = max(2, min(m, n_k - 1))               # can't exceed the number of available K's
    top_idx = list(range(n_k - m_eff, n_k))         # indices of the top-m_eff largest K's
    # Algorithm 1 constructs v=(m_eff,...,1). Positive rank agreement with v
    # means the internal score decreases as K increases in the upper K window.
    ideal = np.arange(m_eff, 0, -1, dtype=float)    # ideal strictly-decreasing reference [m_eff, ..., 1]
    c_vals = {}
    for r in avail:
        # C(candidate) = Spearman(score(K) over K_top, v). This is an internal
        # shape diagnostic; it does not compare the score with ACC or reference labels.
        obs = np.asarray(sil_dict[r], dtype=float)[top_idx]
        rho = spearmanr_safe(obs, ideal)
        # An undefined correlation receives -2, below the valid range [-1,1].
        # If every candidate is undefined, the fallback below returns the smallest one.
        c_vals[r] = rho if not np.isnan(rho) else -2.0   # undefined candidates rank below every valid correlation
    # Threshold branch of Algorithm 1: choose the FIRST (smallest) candidate
    # with C >= tau, rather than maximizing an external agreement score.
    r_star = next((r for r in avail if c_vals[r] >= threshold), None)
    if r_star is None:
        # Fallback branch: maximize C when the threshold is unmet. Sorted candidate
        # order makes the smallest candidate win an exact tie.
        r_star = max(avail, key=lambda r: c_vals[r])
    return r_star


def max_K_eval(k_star):
    """Return K-hat = max(25, round(1.75 K*)) for the benchmark evaluation grid.
    
    Reported rank correlations use candidate counts K=2,...,K-hat. K*
    sets this experimental range; Algorithm 1 then selects from internal
    score curves without comparing them with the reference labels."""
    # K* specifies the benchmark range only. Each K is a requested partition size;
    # the experiment assesses its ranking and does not return an estimated best K.
    return int(max(25, round(1.75 * k_star)))


def max_K_select(k_star):
    """Add ten candidate cluster counts for Algorithm 1's diagnostic window.
    
    Scores at these extra K values help select the neighborhood or embedding
    parameter; they do not enter the reported external rank correlations."""
    # Algorithm 1 sets K_max = K_hat + b, using b=10 in the main comparisons.
    # The extra partitions help select r; reported agreement uses K <= K_hat.
    return max_K_eval(k_star) + 10


## Settings

In [ ]:
K_STAR        = 10       # true number of Gaussian clusters (ground truth K*), matching gaussian_simulation.ipynb
N_REPS        = 100      # number of independent dataset replicates to average over
R_VALUES      = list(range(5, 135, 5))   # full grid: 5,10,...,130 (used for simulation & r* selection)
R_VALUES_PLOT = [5, 10, 30, 50, 100]     # subset of R_VALUES actually shown as boxplot groups in the figure

# Selection notation: code M corresponds to window size s in Algorithm 1;
# THRESHOLD corresponds to tau. Neither is the number of clusters K.
M         = 35     # window size for select_r_topK_spearman's top-m monotonic-decrease criterion
THRESHOLD = 0.95    # minimum Spearman rho (vs. the ideal decreasing sequence) required to accept an r

MAX_K_SELECT = max_K_select(K_STAR)                # K-range upper bound used for r* SELECTION (with +10 buffer)
MAX_K_EVAL   = max_K_eval(K_STAR)                  # K-range upper bound used for every reported/evaluated rho
# K indexes alternative partitions of the same data. Sweep through K_hat+b
# for selection, then restrict to the evaluation subset when reporting rho.
K_TEST       = list(range(2, MAX_K_SELECT + 1))    # K sweep for clustering & r* selection window (the wider range)
EVAL_IDX     = [i for i, K in enumerate(K_TEST) if K <= MAX_K_EVAL]  # indices into K_TEST restricting to the narrower evaluation range

print(f'True K = {K_STAR},  reps = {N_REPS}')
print(f'r grid (simulation): {R_VALUES}')
print(f'r grid (plot):       {R_VALUES_PLOT}')
print(f'K sweep (select): {K_TEST[0]}..{K_TEST[-1]} ({len(K_TEST)} values), m={M}')
print(f'K eval scope:     {K_TEST[0]}..{MAX_K_EVAL} ({len(EVAL_IDX)} values)')


## Evaluation function

In [ ]:
def evaluate_one_rep(X, y_true, K_test, eval_idx, r, random_state=42,
                     labels_by_K=None, baseline=None):
    """Evaluate ordinary and rNN Silhouette for one dataset and candidate r.
    
    Reuse the original-data partitions and ordinary-score/external-criterion
    baseline across r values. Return six Spearman correlations, ordered as
    ordinary/rNN for ACC, ARI, and NMI, followed by the full ordinary and rNN
    score curves and the ACC/ARI/NMI arrays. The full curves support Algorithm 1;
    only eval_idx enters the reported correlations."""
    # Ordinary Silhouette uses Euclidean distances; rNN-Sil uses binary d_r.
    # Both average the same pointwise Silhouette formula over the same partition.
    D_hard = rnn_affinity_distance(X, r=r)

    if labels_by_K is None:
        # Fit each candidate partition once. All indices and all r values below
        # rank these same partitions within this one clustering algorithm.
        labels_by_K = {K: KMeans(n_clusters=K, random_state=random_state,
                                      n_init='auto').fit_predict(X)
                       for K in K_test}

    compute_baseline = baseline is None
    if compute_baseline:
        sil_orig, acc_vals, ari_vals, nmi_vals = [], [], [], []
    else:
        # Reuse the ordinary-silhouette / external-metric arrays computed
        # on an earlier r for this same replicate -- they are independent
        # of r, so recomputing them for every r would be wasted work.
        sil_orig, acc_vals, ari_vals, nmi_vals = [np.asarray(v) for v in baseline]
    sil_hard = []

    for K in K_test:
        labels = labels_by_K[K]
        if len(np.unique(labels)) > 1:
            if compute_baseline:
                sil_orig.append(silhouette_score(X, labels, metric='euclidean'))
            sil_hard.append(silhouette_score(D_hard, labels, metric='precomputed'))
        else:
            # Degenerate clustering (single unique label) -- silhouette
            # undefined; record NaN, masked out later by spearmanr_safe.
            if compute_baseline:
                sil_orig.append(np.nan)
            sil_hard.append(np.nan)
        if compute_baseline:
            acc_vals.append(clustering_accuracy(y_true, labels))
            ari_vals.append(adjusted_rand_score(y_true, labels))
            nmi_vals.append(normalized_mutual_info_score(y_true, labels))

    sil_orig = np.asarray(sil_orig)
    sil_hard = np.array(sil_hard)
    acc_vals = np.asarray(acc_vals)
    ari_vals = np.asarray(ari_vals)
    nmi_vals = np.asarray(nmi_vals)

    def _rho(sil, ref):
        # Spearman correlation restricted to the evaluation-range indices
        # (eval_idx), i.e. K = 2..MAX_K_EVAL, even though sil/ref were
        # computed over the wider K_test (2..MAX_K_SELECT).
        """Compare the internal and external rankings only over the reported evaluation K range."""
        sil_e, ref_e = sil[eval_idx], ref[eval_idx]
        mask = ~np.isnan(sil_e) & ~np.isnan(ref_e)
        return spearmanr(sil_e[mask], ref_e[mask])[0] if mask.sum() >= 2 else np.nan

    return (_rho(sil_orig, acc_vals), _rho(sil_hard, acc_vals), _rho(sil_orig, ari_vals), _rho(sil_hard, ari_vals), _rho(sil_orig, nmi_vals), _rho(sil_hard, nmi_vals), sil_orig, sil_hard, acc_vals, ari_vals, nmi_vals)

## Main repetition loop

In [ ]:
# Generate each replicate and reuse its cluster assignments across r.
# Select r from the internal curves, then compare rNN and ordinary
# Silhouette correlations on the evaluation grid.
from collections import Counter

all_reps_diff = {r: {'acc': {'hard': []}, 'ari': {'hard': []}, 'nmi': {'hard': []}} for r in R_VALUES}
all_reps_diff['r_star'] = {'acc': {'hard': []}, 'ari': {'hard': []}, 'nmi': {'hard': []}}
r_star_log = {'hard': []}   # selected r* per rep

for rep in range(N_REPS):
    if (rep + 1) % 10 == 0:
        print(f'Rep {rep+1}/{N_REPS}')

    # Each replicate uses rep as its random_state, so the N_REPS datasets
    # are independent draws from the same generative model, then
    # standardized (mean 0 / unit variance per feature) before clustering.
    X_rep, y_rep = simulate_gaussian_data(
        n_samples=2000, n_features=20, K_star=K_STAR, random_state=rep)
    # Preprocess each feature as (x - sample mean)/sample standard deviation
    # (StandardScaler uses ddof=0 and leaves constant features at zero).
    # Use the same transformed X for clustering and every original-data score.
    X_rep = StandardScaler().fit_transform(X_rep)

    # K-means labels and ordinary/reference metrics do not depend on r.
    # Compute them once per repetition and reuse them for all r candidates.
    # Fit each candidate partition once. All indices and all r values below
    # rank these same partitions within this one clustering algorithm.
    labels_by_K = {K: KMeans(n_clusters=K, random_state=rep,
                                  n_init='auto').fit_predict(X_rep)
                   for K in K_TEST}
    baseline = None

    # For partition C_K, a_i is mean d_r within its cluster (excluding self),
    # b_i is the smallest mean d_r to another cluster, and
    # S_r(K) = mean_i[(b_i-a_i)/max(a_i,b_i)], with 0 for a_i=b_i=0 or a singleton.
    # Vary r to change d_r while retaining C_K; each array scores one r across K.
    sil_hard_by_r = {}
    sil_orig_rep = acc_vals_rep = ari_vals_rep = nmi_vals_rep = None

    for r in R_VALUES:
        (rho_orig_acc, rho_hard_acc, rho_orig_ari, rho_hard_ari, rho_orig_nmi, rho_hard_nmi, sil_orig, sil_hard, acc_vals, ari_vals, nmi_vals) = evaluate_one_rep(
            X_rep, y_rep, K_TEST, EVAL_IDX, r=r, random_state=rep,
            labels_by_K=labels_by_K, baseline=baseline)

        # "Improvement" = rho(rNN-based silhouette, external metric) minus
        # rho(ordinary silhouette, external metric), for this rep and this r.
        all_reps_diff[r]['acc']['hard'].append(rho_hard_acc - rho_orig_acc)
        all_reps_diff[r]['ari']['hard'].append(rho_hard_ari - rho_orig_ari)
        all_reps_diff[r]['nmi']['hard'].append(rho_hard_nmi - rho_orig_nmi)

        sil_hard_by_r[r] = sil_hard
        # sil_orig / acc / ari / nmi don't depend on r — capture once
        if sil_orig_rep is None:
            sil_orig_rep = sil_orig
            acc_vals_rep  = acc_vals
            ari_vals_rep  = ari_vals
            nmi_vals_rep  = nmi_vals
            baseline = (sil_orig_rep, acc_vals_rep, ari_vals_rep, nmi_vals_rep)

    # Baseline (ordinary-silhouette) rho's for this rep, evaluated once
    # over EVAL_IDX -- used below as the reference the r*-selected curve's
    # improvement is measured against.
    rho_orig_acc_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], acc_vals_rep[EVAL_IDX])
    rho_orig_ari_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], ari_vals_rep[EVAL_IDX])
    rho_orig_nmi_base = spearmanr_safe(sil_orig_rep[EVAL_IDX], nmi_vals_rep[EVAL_IDX])

    # -- Select r* per rep: select_r_topK_spearman, m=35, over the WIDE K_TEST --
    # Selection is unsupervised w.r.t. the external metrics (it only looks
    # at the silhouette curves' own shape), so the same r* is used for all
    # three external metrics within a given rep/variant.
    r_star_h = select_r_topK_spearman(sil_hard_by_r, R_VALUES, m=M, n_k=len(K_TEST), threshold=THRESHOLD)
    r_star_log['hard'].append(r_star_h)

    all_reps_diff['r_star']['acc']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], acc_vals_rep[EVAL_IDX]) - rho_orig_acc_base)
    all_reps_diff['r_star']['ari']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], ari_vals_rep[EVAL_IDX]) - rho_orig_ari_base)
    all_reps_diff['r_star']['nmi']['hard'].append(
        spearmanr_safe(sil_hard_by_r[r_star_h][EVAL_IDX], nmi_vals_rep[EVAL_IDX]) - rho_orig_nmi_base)

print('\nAll repetitions completed!')
print('\nr* selection frequency:')
for variant in ['hard']:
    counts = Counter(r_star_log[variant])
    print(f'  {variant}: ' + ', '.join(f'r={r}: {counts[r]}x' for r in sorted(counts)))

# -- Summary: mean +/- std improvement across all N_REPS reps, for every
# fixed r and for the r*-selected curve. This is the numeric backbone of
# the boxplots produced later in this notebook.
print()
for r in R_VALUES:
    d_ha = np.array(all_reps_diff[r]['acc']['hard'])
    d_hr = np.array(all_reps_diff[r]['ari']['hard'])
    d_hn = np.array(all_reps_diff[r]['nmi']['hard'])
    print(f'  r={r:3d}  Hard(ACC): {d_ha.mean():.4f}±{d_ha.std():.4f}  Hard(ARI): {d_hr.mean():.4f}±{d_hr.std():.4f}  Hard(NMI): {d_hn.mean():.4f}±{d_hn.std():.4f}')

d_ha = np.array(all_reps_diff['r_star']['acc']['hard'])
d_hr = np.array(all_reps_diff['r_star']['ari']['hard'])
d_hn = np.array(all_reps_diff['r_star']['nmi']['hard'])
print(f'  r*  Hard(ACC): {d_ha.mean():.4f}±{d_ha.std():.4f}  Hard(ARI): {d_hr.mean():.4f}±{d_hr.std():.4f}  Hard(NMI): {d_hn.mean():.4f}±{d_hn.std():.4f}')

In [ ]:
# Cache the full 100-replicate results (and the per-rep selected r* log) so
# they can be reloaded below without rerunning the (expensive) main loop,
# and so combined_boxplot_figure.ipynb can reuse this exact pickle for the
# bottom panel of its combined figure.
import pickle, os

os.makedirs('data', exist_ok=True)
with open('data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl', 'wb') as f:
    pickle.dump({
        'all_reps_diff':  all_reps_diff,
        'R_VALUES':       R_VALUES,
        'R_VALUES_PLOT':  R_VALUES_PLOT,
        'K_TEST':         K_TEST,
        'K_STAR':         K_STAR,
        'N_REPS':         N_REPS,
        'MAX_K_SELECT':   MAX_K_SELECT,
        'MAX_K_EVAL':     MAX_K_EVAL,
        'M':              M,
        'r_star_log':     r_star_log,
    }, f)
print('Results saved to data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl')

## Boxplot

## Read the saved replication results for plotting

When running interactively, this cell can load an existing cache without
rerunning the replication loop. The command-line runner executes all code cells.


In [ ]:
# Reload the cached 100-rep results (lets the rest of the notebook -- the
# frequency table and boxplot cells below -- be re-run without repeating
# the expensive main loop above).
import pickle
from collections import Counter

with open('data/gaussian_simulation_boxplot_paperkmax_buffer10_nonself_results.pkl', 'rb') as f:
    saved = pickle.load(f)

all_reps_diff = saved['all_reps_diff']
R_VALUES      = saved['R_VALUES']
R_VALUES_PLOT = saved.get('R_VALUES_PLOT', [5, 10, 30, 50, 100])
K_TEST        = saved['K_TEST']
K_STAR        = saved['K_STAR']
N_REPS        = saved['N_REPS']
MAX_K_SELECT  = saved.get('MAX_K_SELECT')
MAX_K_EVAL    = saved.get('MAX_K_EVAL')
# Selection notation: code M corresponds to window size s in Algorithm 1;
# THRESHOLD corresponds to tau. Neither is the number of clusters K.
M             = saved.get('M', 35)
r_star_log    = saved.get('r_star_log', {'hard': []})

print(f'Loaded: K_STAR={K_STAR}, N_REPS={N_REPS}')
print(f'r grid (simulation): {R_VALUES}')
print(f'r grid (plot):       {R_VALUES_PLOT}')
print(f'K sweep (select): {K_TEST}')
print(f'max_K_select={MAX_K_SELECT}  max_K_eval={MAX_K_EVAL}  m={M}')
if any((r_star_log[v] for v in ('hard',))):
    print('\nr* selection frequency:')
    for variant in ['hard']:
        counts = Counter(r_star_log[variant])
        print(f'  {variant}: ' + ', '.join(f'r={r}: {counts[r]}x' for r in sorted(counts)))

In [ ]:
# -- Frequency table of selected r* values --
# Shows, across the 100 replicates, how often each candidate r in R_VALUES
# was the one picked by select_r_topK_spearman as r* (separately for the
# Hard variants). Useful for sanity-checking that r* selection is
# concentrating on a narrow, sensible band of r rather than scattering
# widely across the grid.
from collections import Counter

variants   = ['hard']
var_labels = {'hard': 'Neighborhood Hard'}
col_w = 12

print(f"{'r':>6}  " + "  ".join(f"{var_labels[v]:>{col_w}}" for v in variants))
print("-" * (8 + (col_w + 2) * len(variants)))
for r in R_VALUES:
    row = f"{r:>6}  "
    for v in variants:
        counts = Counter(r_star_log[v])
        n      = counts.get(r, 0)
        pct    = 100 * n / len(r_star_log[v]) if r_star_log[v] else 0
        row   += f"{f'{n} ({pct:.0f}%)':>{col_w}}  "
    print(row)
print("-" * (8 + (col_w + 2) * len(variants)))
total_row = f"{'Total':>6}  "
for v in variants:
    total_row += f"{len(r_star_log[v]):>{col_w}}  "
print(total_row)

In [ ]:
def plot_one_panel(ax, all_reps_diff, r_values, ref_key, ext_label):
    """Draw one replicated-improvement panel, including Figure 2(b).
    
    Each stored value is Delta rho = rho(rNN-Sil, external criterion)
    minus rho(Sil, external criterion) for one generated dataset. Group
    fixed r values first, then the r* selected separately in each replicate
    by Algorithm 1. Boxes summarize replication variability; positive
    values mean better rank agreement than ordinary Silhouette.
    Return representative patches and labels for the caller's legend."""
    # The retained "hard" key refers to binary-neighbor rNN-Sil results.
    # There is one box per group, so the series is centered with zero offset.
    variants = ['hard']
    colors   = {'hard': 'steelblue'}
    labels_legend = {'hard': 'Silhouette Score (Neighborhood Hard)'}

    width = 0.28
    offsets_rel = [0.0]

    r_star_keys   = ['r_star']
    r_star_labels = [r'$r^*$']

    # Keep the fixed-r groups in the requested display order and append selected r*.
    # The r* group may use a different neighborhood size in every replication.
    plot_keys   = list(r_values) + r_star_keys
    plot_labels = [f'r = {r}' for r in r_values] + r_star_labels

    x_ticks, x_tick_labels, proxy_patches = [], [], {}

    for i, (rkey, rlabel) in enumerate(zip(plot_keys, plot_labels)):
        center = i * (2 * width + 0.5)
        is_r_star = rkey in r_star_keys
        for variant, offset in zip(variants, offsets_rel):
            # This list contains one paired Delta rho per replicated dataset for ref_key
            # (ACC, ARI, or NMI). The subtraction happened before plotting, on matching partitions.
            data = all_reps_diff[rkey][ref_key][variant]
            pos  = center + offset
            # Box: 25th--75th percentiles; black line: median. Default whiskers extend
            # to the furthest observations within 1.5 IQR, with remaining points shown as fliers.
            # These show replication variability, not confidence intervals for the mean.
            bp = ax.boxplot(
                [data], positions=[pos], widths=width * 0.9,
                patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                manage_ticks=False,
            )
            patch = bp['boxes'][0]
            patch.set_facecolor(colors[variant])
            patch.set_alpha(0.85 if is_r_star else 0.65)
            # Hatching distinguishes the label-free selected-r procedure from a fixed r.
            # It does not indicate statistical significance or an oracle choice.
            if is_r_star:
                patch.set_hatch('///')
            # Retain one patch per series so the caller can build a single shared legend
            # without adding an entry for every neighborhood group.
            if variant not in proxy_patches:
                proxy_patches[variant] = patch
        x_ticks.append(center)
        x_tick_labels.append(rlabel)

    # Vertical separator before the r* group
    sep_x = (len(r_values) - 1) * (2 * width + 0.5) + (2 * width + 0.5) * 0.75
    ax.axvline(sep_x, color='gray', linestyle=':', linewidth=1.5, alpha=0.6)

    # Highlight the r* group with a red-edged box
    pad     = width + 0.3
    left_x  = x_ticks[len(r_values)] - pad
    right_x = x_ticks[-1] + pad
    ax.axvspan(left_x, right_x, facecolor='none', edgecolor='red',
              linewidth=2.5, zorder=10)

    # Zero means equal rank agreement; points above it favor rNN-Sil over Sil.
    # This is a difference of correlations, not a difference of raw index values.
    ax.axhline(0, color='grey', linewidth=1, linestyle='--')
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_tick_labels, fontsize=20)
    ax.tick_params(axis='y', labelsize=16)
    ax.set_xlabel('')
    ax.set_ylabel(
        f'Difference in Spearman $\\rho$ with {ext_label}',
        fontsize=18)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_title(ext_label, fontsize=29, fontweight='bold')

    return proxy_patches, labels_legend


def plot_diff_boxplot_single(all_reps_diff, r_values_plot, K_star, n_reps,
                             ref_key, ext_label, file_tag):
    """Save a standalone Case 1 improvement panel for ACC, ARI, or NMI.
    
    Use the same replicated differences and selected-r group as Figure 2(b).
    The helper below draws the boxes; this wrapper adds the experiment
    description, common legend, and PDF/PNG exports."""
    # Allocate one group for each displayed fixed r plus the selected-r group;
    # scale the canvas width so adding a group does not crowd the labels.
    n_groups = len(r_values_plot) + 1
    fig, ax = plt.subplots(1, 1, figsize=(1.4 * n_groups + 2, 7))

    proxy_patches, labels_legend = plot_one_panel(
        ax, all_reps_diff, r_values_plot, ref_key, ext_label)

    fig.suptitle(
        f'Simulated Gaussian Dataset: Improvement of $r$NN Affinity Silhouette over Silhouette ({ext_label})\n'
        f'(True $\\mathbf{{K}} = {K_star}$, {n_reps} reps, eval $k_{{max}}$=max(25,1.75$k^*$), select +10, m=35)',
        fontsize=24, fontweight='bold',
    )

    # Use the panel's representative patch so the shared legend matches the boxes.
    handles = [proxy_patches[v] for v in ('hard',)]
    labels  = [labels_legend[v] for v in ('hard',)]
    fig.legend(
        handles, labels,
        fontsize=20,
        loc='lower center',
        bbox_to_anchor=(0.5, -0.1),
        ncol=2,
        framealpha=0.9,
    )

    # Reserve space for the external legend before exporting the same panel
    # as a vector PDF and a raster PNG preview.
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.24)
    import os
    os.makedirs('figures/smoothed_peak_gap', exist_ok=True)
    plt.savefig(f'figures/smoothed_peak_gap/SimGa_diff_boxplot_{file_tag}.pdf', bbox_inches='tight')
    plt.savefig(f'figures/smoothed_peak_gap/SimGa_diff_boxplot_{file_tag}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Produce one standalone figure per external metric (ACC, ARI, NMI), each
# showing R_VALUES_PLOT boxplot groups plus the r* group.
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'acc', 'ACC', 'acc_paperkmax_buffer10')
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'ari', 'ARI', 'ari_paperkmax_buffer10')
plot_diff_boxplot_single(all_reps_diff, R_VALUES_PLOT, K_STAR, N_REPS, 'nmi', 'NMI', 'nmi_paperkmax_buffer10')